# Notebook 09 — Stage L Interpretability, Ethics, and Clinical Trust

Purpose:
- build a trust-layer package on top of Stage J/K artifacts
- generate global/local explanations with graceful fallbacks
- quantify subgroup disparities and create a failure-case catalog
- export Stage L model-card inputs and reproducibility artifacts

In [1]:
from pathlib import Path
import os
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook09_stage_l'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook09_stage_l'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook09_stage_l'
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STAGE_L_MAX_ROWS = int(os.getenv('STAGE_L_MAX_ROWS', '60000'))
STAGE_L_LOCAL_EXPLAIN_N = int(os.getenv('STAGE_L_LOCAL_EXPLAIN_N', '2000'))

print('Notebook 09 workspace ready')
print('Crash-prevention caps -> STAGE_L_MAX_ROWS:', STAGE_L_MAX_ROWS, '| STAGE_L_LOCAL_EXPLAIN_N:', STAGE_L_LOCAL_EXPLAIN_N)

Notebook 09 workspace ready
Crash-prevention caps -> STAGE_L_MAX_ROWS: 60000 | STAGE_L_LOCAL_EXPLAIN_N: 2000


In [11]:
# Stage J/K handoff + Stage L canonical trust dataset
phase_j_manifest_path = META_DIR / 'phase_j_manifest.json'
phase_k_manifest_path = META_DIR / 'phase_k_manifest.json'
if not phase_j_manifest_path.exists():
    raise FileNotFoundError(f'Missing Stage J manifest: {phase_j_manifest_path}')
if not phase_k_manifest_path.exists():
    raise FileNotFoundError(f'Missing Stage K manifest: {phase_k_manifest_path}')

with open(phase_j_manifest_path, 'r', encoding='utf-8') as f:
    phase_j_manifest = json.load(f)
with open(phase_k_manifest_path, 'r', encoding='utf-8') as f:
    phase_k_manifest = json.load(f)

selected_j = phase_j_manifest.get('selected_model', 'unknown')
print('Stage J selected model context:', selected_j)

panel_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
cols = [
    'patient_id', 'day', 'I_stage_f_base', 'hazard_prob_stage_f', 'stage_f_escalation_event',
    'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active'
]
data = pd.read_parquet(panel_path, columns=cols).copy()
sample_n = min(STAGE_L_MAX_ROWS, len(data))
df = data.sample(n=sample_n, random_state=42).reset_index(drop=True)

for c in ['I_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start']:
    df[f'{c}__is_missing'] = df[c].isna().astype(np.int8)

df['is_nonresponse'] = df['response_state_active'].eq('nonresponse').astype(np.int8)
df['is_partial'] = df['response_state_active'].eq('partial_response').astype(np.int8)
df['is_stabilized'] = df['response_state_active'].eq('stabilized').astype(np.int8)
df['I_stage_f_base_x_cycle'] = df['I_stage_f_base'].fillna(df['I_stage_f_base'].median()) * df['cycle_id_stage_f']

feature_set = [
    'cycle_id_stage_f', 'months_since_cycle_start', 'I_stage_f_base', 'hazard_prob_stage_f',
    'is_nonresponse', 'is_partial', 'is_stabilized',
    'I_stage_f_base__is_missing', 'hazard_prob_stage_f__is_missing', 'months_since_cycle_start__is_missing',
    'I_stage_f_base_x_cycle'
]

X = df[feature_set].copy()
y = df['stage_f_escalation_event'].astype(int).to_numpy()

split_tag = df['patient_id'].astype(str).to_numpy()
X_train, X_test, y_train, y_test, split_train, split_test = train_test_split(
    X, y, split_tag, test_size=0.25, random_state=42, stratify=y
)

print('Stage L trust dataset shape:', X.shape)
print('Escalation prevalence:', float(y.mean()))

Stage J selected model context: hist_gb_strong
Stage L trust dataset shape: (60000, 11)
Escalation prevalence: 0.011983333333333334


In [12]:
# Schema reconnaissance for multi-diagnosis / medication logic
panel_full = pd.read_parquet(panel_path)
all_cols = panel_full.columns.tolist()
diagnosis_cols = [c for c in all_cols if ('diagnos' in c.lower()) or ('dx' in c.lower())]
medication_cols = [c for c in all_cols if ('med' in c.lower()) or ('drug' in c.lower()) or ('treat' in c.lower())]
registry_cols = [c for c in all_cols if ('registry' in c.lower()) or ('source' in c.lower())]

diag_path = PROJECT_ROOT / 'Data' / 'diagnosis_history.parquet'
med_path = PROJECT_ROOT / 'Data' / 'medication_event.parquet'
diag_df = pd.read_parquet(diag_path) if diag_path.exists() else pd.DataFrame()
med_df = pd.read_parquet(med_path) if med_path.exists() else pd.DataFrame()

print('Panel shape:', panel_full.shape)
print('Diagnosis-like columns found in phase_f panel:', len(diagnosis_cols))
print('Medication-like columns found in phase_f panel:', len(medication_cols))
print('Registry/source-like columns found:', len(registry_cols))
print('Sample diagnosis columns in phase_f panel:', diagnosis_cols[:15])
print('Sample medication columns in phase_f panel:', medication_cols[:15])
print('Diagnosis history exists:', diag_path.exists(), '| rows:', len(diag_df), '| cols:', list(diag_df.columns[:15]))
print('Medication events exist:', med_path.exists(), '| rows:', len(med_df), '| cols:', list(med_df.columns[:15]))

panel_full.head(2)

Panel shape: (4900000, 12)
Diagnosis-like columns found in phase_f panel: 0
Medication-like columns found in phase_f panel: 0
Registry/source-like columns found: 0
Sample diagnosis columns in phase_f panel: []
Sample medication columns in phase_f panel: []
Diagnosis history exists: True | rows: 160280 | cols: ['patient_id', 'day', 'diagnosis_code']
Medication events exist: True | rows: 729490 | cols: ['patient_id', 'event_day', 'diagnosis_code', 'instability', 'medication_class', 'medication_name', 'regimen_phase', 'strategy_code', 'medication_class_group', 'enrichment_version']


,patient_id,day,I_phase_b,I_stage_f_base,hazard_prob,hazard_prob_stage_f_base,hazard_prob_stage_f,baseline_admission_event,stage_f_escalation_event,cycle_id_stage_f,months_since_cycle_start,response_state_active
0,P000000,0,0.399309,0.399309,0.003945,0.003945,0.003945,0,0,0,0,pre_admission
1,P000000,30,0.219145,0.219145,0.003946,0.003946,0.003946,0,0,0,1,pre_admission


In [4]:
# Dual-model trust comparison (intrinsic + strong tabular proxy)
linear_pipe = Pipeline(steps=[
    ('imp', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1200, class_weight='balanced', random_state=42))
])
tree_pipe = Pipeline(steps=[
    ('imp', SimpleImputer(strategy='median')),
    ('clf', HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06, max_iter=260, random_state=42))
])

linear_pipe.fit(X_train, y_train)
tree_pipe.fit(X_train, y_train)

proba_lin = linear_pipe.predict_proba(X_test)[:, 1]
proba_tree = tree_pipe.predict_proba(X_test)[:, 1]

metrics_df = pd.DataFrame([
    {
        'model': 'logistic_intrinsic',
        'auroc': float(roc_auc_score(y_test, proba_lin)),
        'pr_auc': float(average_precision_score(y_test, proba_lin)),
        'brier': float(brier_score_loss(y_test, proba_lin)),
        'log_loss': float(log_loss(y_test, proba_lin, labels=[0, 1]))
    },
    {
        'model': 'hist_gb_proxy',
        'auroc': float(roc_auc_score(y_test, proba_tree)),
        'pr_auc': float(average_precision_score(y_test, proba_tree)),
        'brier': float(brier_score_loss(y_test, proba_tree)),
        'log_loss': float(log_loss(y_test, proba_tree, labels=[0, 1]))
    }
])
metrics_df.to_csv(TABLE_DIR / 'stage_l_model_metrics.csv', index=False)

coef = linear_pipe.named_steps['clf'].coef_.ravel()
coef_df = pd.DataFrame({'feature': feature_set, 'coef': coef, 'abs_coef': np.abs(coef)}).sort_values('abs_coef', ascending=False)
coef_df.to_csv(TABLE_DIR / 'stage_l_intrinsic_alignment_coefficients.csv', index=False)

print('Stage L base trust metrics/artifacts generated')
metrics_df

Stage L base trust metrics/artifacts generated


,model,auroc,pr_auc,brier,log_loss
0,logistic_intrinsic,0.999942,0.997365,5.226191e-03,1.475882e-02
1,hist_gb_proxy,1.000000,1.000000,6.791150e-13,1.516075e-07


In [5]:
# Global/local explainability: SHAP-first (tree/linear/permutation) with robust fallback
X_test_imp_tree = pd.DataFrame(tree_pipe.named_steps['imp'].transform(X_test), columns=feature_set)
X_test_imp_lin = pd.DataFrame(linear_pipe.named_steps['imp'].transform(X_test), columns=feature_set)

explain_method = 'fallback'
global_imp_df = None
local_exp_df = None
shap_errors = []

try:
    import shap

    # path A: tree SHAP on HistGradientBoosting
    try:
        explain_method = 'shap_tree'
        tree_model = tree_pipe.named_steps['clf']
        tree_explainer = shap.Explainer(tree_model)
        shap_values_tree = tree_explainer(X_test_imp_tree.iloc[:STAGE_L_LOCAL_EXPLAIN_N])

        abs_mean = np.abs(shap_values_tree.values).mean(axis=0)
        global_imp_df = pd.DataFrame({'feature': feature_set, 'importance': abs_mean}).sort_values('importance', ascending=False)

        local_block = pd.DataFrame(shap_values_tree.values, columns=feature_set)
        local_block['sample_index'] = np.arange(len(local_block))
        local_exp_df = local_block.melt(id_vars=['sample_index'], var_name='feature', value_name='contribution')

    except Exception as e_tree:
        shap_errors.append(f'tree_path: {e_tree}')

    # path B: linear SHAP on logistic regression
    if global_imp_df is None or local_exp_df is None:
        try:
            explain_method = 'shap_linear'
            lin_model = linear_pipe.named_steps['clf']
            x_local = X_test_imp_lin.iloc[:STAGE_L_LOCAL_EXPLAIN_N]
            lin_explainer = shap.LinearExplainer(lin_model, x_local)
            shap_values_lin = lin_explainer.shap_values(x_local)

            abs_mean = np.abs(shap_values_lin).mean(axis=0)
            global_imp_df = pd.DataFrame({'feature': feature_set, 'importance': abs_mean}).sort_values('importance', ascending=False)

            local_block = pd.DataFrame(shap_values_lin, columns=feature_set)
            local_block['sample_index'] = np.arange(len(local_block))
            local_exp_df = local_block.melt(id_vars=['sample_index'], var_name='feature', value_name='contribution')
        except Exception as e_linear:
            shap_errors.append(f'linear_path: {e_linear}')

    # path C: model-agnostic SHAP permutation explainer
    if global_imp_df is None or local_exp_df is None:
        try:
            explain_method = 'shap_permutation'
            n_bg = min(120, len(X_train))
            n_local = min(400, STAGE_L_LOCAL_EXPLAIN_N, len(X_test_imp_tree))
            x_bg = X_train.iloc[:n_bg].copy()
            x_local = X_test.iloc[:n_local].copy()

            def _predict_fn(x):
                x_df = pd.DataFrame(x, columns=feature_set)
                return tree_pipe.predict_proba(x_df)[:, 1]

            perm_explainer = shap.Explainer(_predict_fn, x_bg, algorithm='permutation')
            shap_values_perm = perm_explainer(x_local)

            abs_mean = np.abs(shap_values_perm.values).mean(axis=0)
            global_imp_df = pd.DataFrame({'feature': feature_set, 'importance': abs_mean}).sort_values('importance', ascending=False)

            local_block = pd.DataFrame(shap_values_perm.values, columns=feature_set)
            local_block['sample_index'] = np.arange(len(local_block))
            local_exp_df = local_block.melt(id_vars=['sample_index'], var_name='feature', value_name='contribution')
        except Exception as e_perm:
            shap_errors.append(f'permutation_path: {e_perm}')

except Exception as e_import:
    shap_errors.append(f'import_path: {e_import}')

if (global_imp_df is None) or (local_exp_df is None):
    explain_method = 'fallback'
    try:
        importances = tree_pipe.named_steps['clf'].feature_importances_
    except Exception:
        importances = np.abs(linear_pipe.named_steps['clf'].coef_.ravel())

    global_imp_df = pd.DataFrame({'feature': feature_set, 'importance': importances}).sort_values('importance', ascending=False)

    scaler = linear_pipe.named_steps['scale']
    x_scaled = scaler.transform(X_test_imp_lin.iloc[:STAGE_L_LOCAL_EXPLAIN_N].to_numpy())
    contrib = x_scaled * linear_pipe.named_steps['clf'].coef_.reshape(1, -1)
    local_block = pd.DataFrame(contrib, columns=feature_set)
    local_block['sample_index'] = np.arange(len(local_block))
    local_exp_df = local_block.melt(id_vars=['sample_index'], var_name='feature', value_name='contribution')

global_imp_df.to_csv(TABLE_DIR / 'stage_l_global_explanations.csv', index=False)
local_exp_df.to_csv(TABLE_DIR / 'stage_l_local_explanations_long.csv', index=False)

topn = global_imp_df.head(15).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(topn['feature'], topn['importance'])
plt.title(f'Stage L Global Feature Importance ({explain_method})')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_l_global_explanation_barh.png', dpi=140, bbox_inches='tight')
plt.close()

with open(REPORT_DIR / 'stage_l_explainability_runtime_log.txt', 'w', encoding='utf-8') as f:
    f.write(f'explain_method: {explain_method}\n')
    if len(shap_errors) == 0:
        f.write('shap_errors: none\n')
    else:
        f.write('shap_errors:\n')
        for err in shap_errors:
            f.write(f'- {err}\n')

print('Explanation method used:', explain_method)
print('Global/local explanation artifacts generated')

c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Explanation method used: shap_tree
Global/local explanation artifacts generated


In [6]:
# Fairness subgroup audit + failure-case catalog + counterfactual nearest-opposite examples
pred_tree = (proba_tree >= 0.5).astype(int)
eval_df = X_test.copy().reset_index(drop=True)
eval_df['y_true'] = y_test
eval_df['y_pred'] = pred_tree
eval_df['p_pred'] = proba_tree
eval_df['response_state_active'] = df.loc[X_test.index, 'response_state_active'].to_numpy()

def _safe_rate(numer, denom):
    return float(numer / denom) if denom > 0 else np.nan

group_rows = []
for grp, part in eval_df.groupby('response_state_active'):
    tp = int(((part['y_true'] == 1) & (part['y_pred'] == 1)).sum())
    tn = int(((part['y_true'] == 0) & (part['y_pred'] == 0)).sum())
    fp = int(((part['y_true'] == 0) & (part['y_pred'] == 1)).sum())
    fn = int(((part['y_true'] == 1) & (part['y_pred'] == 0)).sum())
    group_rows.append({
        'group': str(grp),
        'n': int(len(part)),
        'auroc': float(roc_auc_score(part['y_true'], part['p_pred'])) if part['y_true'].nunique() > 1 else np.nan,
        'fpr': _safe_rate(fp, fp + tn),
        'fnr': _safe_rate(fn, fn + tp),
        'ppv': _safe_rate(tp, tp + fp),
        'npv': _safe_rate(tn, tn + fn)
    })

fairness_df = pd.DataFrame(group_rows).sort_values('n', ascending=False)
fairness_df.to_csv(TABLE_DIR / 'stage_l_fairness_subgroup_audit.csv', index=False)

# failure-case catalog: highest confidence errors
errors_df = eval_df[eval_df['y_true'] != eval_df['y_pred']].copy()
errors_df['error_type'] = np.where((errors_df['y_true'] == 1) & (errors_df['y_pred'] == 0), 'FN', 'FP')
errors_df['error_confidence'] = np.where(errors_df['error_type'] == 'FN', 1 - errors_df['p_pred'], errors_df['p_pred'])
failure_catalog = errors_df.sort_values('error_confidence', ascending=False).head(200)
failure_catalog.to_csv(TABLE_DIR / 'stage_l_failure_case_catalog.csv', index=False)

# simple counterfactual set: nearest opposite-label sample in imputed feature space
x_pool = X_test_imp_tree.to_numpy(dtype=np.float32, copy=False)
y_pool = y_test.astype(int)
idx_candidates = np.arange(len(y_pool))
query_idx = idx_candidates[:min(120, len(idx_candidates))]
cf_rows = []
for q in query_idx:
    target_label = 1 - int(y_pool[q])
    opp_idx = idx_candidates[y_pool == target_label]
    if len(opp_idx) == 0:
        continue
    d = ((x_pool[opp_idx] - x_pool[q]) ** 2).sum(axis=1)
    best = int(opp_idx[int(np.argmin(d))])
    cf_rows.append({
        'query_index': int(q),
        'query_true': int(y_pool[q]),
        'counterfactual_index': int(best),
        'counterfactual_true': int(y_pool[best]),
        'distance_l2_sq': float(np.min(d))
    })

counterfactual_df = pd.DataFrame(cf_rows)
counterfactual_df.to_csv(TABLE_DIR / 'stage_l_counterfactual_examples.csv', index=False)

summary = {
    'selected_model_context_from_stage_j': selected_j,
    'trust_model_used_for_audit': 'hist_gb_proxy',
    'explain_method': explain_method,
    'n_eval_rows': int(len(eval_df)),
    'n_failure_cases_cataloged': int(len(failure_catalog)),
    'n_counterfactual_pairs': int(len(counterfactual_df))
}
with open(REPORT_DIR / 'stage_l_clinical_trust_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage L Clinical Trust Summary\n')
    for k, v in summary.items():
        f.write(f'{k}: {v}\n')

gc.collect()
print('Fairness/failure/counterfactual artifacts generated')
fairness_df

Fairness/failure/counterfactual artifacts generated


,group,n,auroc,fpr,fnr,ppv,npv
2,pre_admission,11350,NaN,0.0,NaN,NaN,1.0
3,stabilized,1909,1.0,0.0,0.0,1.0,1.0
1,partial_response,879,1.0,0.0,0.0,1.0,1.0
0,nonresponse,862,1.0,0.0,0.0,1.0,1.0


In [13]:
# Multi-diagnosis progression + medication recommendation by patient category (registry-backed, with inferred fallback)
trajectory_df = panel_full[['patient_id', 'day', 'I_phase_b', 'I_stage_f_base', 'hazard_prob_stage_f', 'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active', 'stage_f_escalation_event']].copy()

diag_path = PROJECT_ROOT / 'Data' / 'diagnosis_history.parquet'
med_path = PROJECT_ROOT / 'Data' / 'medication_event.parquet'
diag_hist = pd.read_parquet(diag_path) if diag_path.exists() else pd.DataFrame()
med_evt = pd.read_parquet(med_path) if med_path.exists() else pd.DataFrame()

diag_used = False
med_used = False

diag_long = None
if len(diag_hist) > 0 and {'patient_id'}.issubset(diag_hist.columns):
    day_col = 'day' if 'day' in diag_hist.columns else ('event_day' if 'event_day' in diag_hist.columns else None)
    code_col = 'diagnosis_code' if 'diagnosis_code' in diag_hist.columns else ('diagnosis' if 'diagnosis' in diag_hist.columns else ('dx_code' if 'dx_code' in diag_hist.columns else None))
    if day_col is not None and code_col is not None:
        diag_long = diag_hist[['patient_id', day_col, code_col]].copy()
        diag_long = diag_long.rename(columns={day_col: 'day', code_col: 'inferred_diagnosis'})
        diag_long['inferred_diagnosis'] = diag_long['inferred_diagnosis'].astype(str)
        diag_used = True

if diag_long is None:
    inst_q = pd.qcut(trajectory_df['I_stage_f_base'].rank(method='first'), q=4, labels=['low', 'mid', 'high', 'very_high'])
    haz_q = pd.qcut(trajectory_df['hazard_prob_stage_f'].rank(method='first'), q=4, labels=['low', 'mid', 'high', 'very_high'])
    trajectory_df['inst_q'] = inst_q.astype(str)
    trajectory_df['haz_q'] = haz_q.astype(str)

    def infer_dx_labels(row):
        labels = []
        if row['response_state_active'] in ['nonresponse', 'partial_response'] and row['inst_q'] in ['high', 'very_high']:
            labels.append('dx_mood_instability')
        if row['haz_q'] in ['high', 'very_high'] and row['months_since_cycle_start'] >= 12:
            labels.append('dx_risk_progression')
        if row['cycle_id_stage_f'] >= 2 and row['response_state_active'] == 'nonresponse':
            labels.append('dx_treatment_resistance')
        if row['response_state_active'] == 'stabilized' and row['inst_q'] in ['low', 'mid'] and row['haz_q'] in ['low', 'mid']:
            labels.append('dx_stabilized_recovery')
        if len(labels) == 0:
            labels.append('dx_general_monitoring')
        return labels

    trajectory_df['inferred_dx_set'] = trajectory_df.apply(infer_dx_labels, axis=1)
    trajectory_df['inferred_dx_count'] = trajectory_df['inferred_dx_set'].apply(len)
    dx_long = trajectory_df[['patient_id', 'day', 'inferred_dx_set', 'inferred_dx_count']].explode('inferred_dx_set').rename(columns={'inferred_dx_set': 'inferred_diagnosis'})
else:
    day_index = trajectory_df[['patient_id', 'day']].copy()
    dx_long = diag_long.merge(day_index, on=['patient_id', 'day'], how='inner')
    dx_cnt = dx_long.groupby(['patient_id', 'day']).size().rename('inferred_dx_count').reset_index()
    trajectory_df = trajectory_df.merge(dx_cnt, on=['patient_id', 'day'], how='left')
    trajectory_df['inferred_dx_count'] = trajectory_df['inferred_dx_count'].fillna(0).astype(int)
    trajectory_df['inferred_dx_set'] = trajectory_df['inferred_dx_count'].apply(lambda n: ['dx_general_monitoring'] if n == 0 else ['dx_from_registry'])

if len(med_evt) > 0 and {'patient_id'}.issubset(med_evt.columns):
    med_day_col = 'event_day' if 'event_day' in med_evt.columns else ('day' if 'day' in med_evt.columns else None)
    med_code_col = None
    for cand in ['medication_name', 'medication_code', 'drug_name', 'drug_class', 'medication_class']:
        if cand in med_evt.columns:
            med_code_col = cand
            break
    if med_day_col is not None and med_code_col is not None:
        med_tmp = med_evt[['patient_id', med_day_col, med_code_col]].copy()
        med_tmp = med_tmp.rename(columns={med_day_col: 'day', med_code_col: 'medication_label'})
        med_tmp['medication_label'] = med_tmp['medication_label'].astype(str)
        med_used = True
    else:
        med_tmp = pd.DataFrame(columns=['patient_id', 'day', 'medication_label'])
else:
    med_tmp = pd.DataFrame(columns=['patient_id', 'day', 'medication_label'])

patient_dx = trajectory_df.groupby('patient_id').agg(
    mean_instability=('I_stage_f_base', 'mean'),
    mean_hazard=('hazard_prob_stage_f', 'mean'),
    max_cycle=('cycle_id_stage_f', 'max'),
    max_months=('months_since_cycle_start', 'max'),
    escalation_rate=('stage_f_escalation_event', 'mean'),
    dx_burden=('inferred_dx_count', 'mean')
).reset_index()

patient_dx['patient_category'] = np.select(
    [
        (patient_dx['escalation_rate'] >= 0.08) | (patient_dx['dx_burden'] >= 2.2),
        (patient_dx['escalation_rate'] >= 0.02) | (patient_dx['dx_burden'] >= 1.4),
    ],
    ['high_complexity', 'moderate_complexity'],
    default='low_complexity'
)

if med_used:
    med_freq = med_tmp.groupby(['patient_id', 'medication_label']).size().rename('n').reset_index()
    med_pref = med_freq.sort_values(['patient_id', 'n'], ascending=[True, False]).drop_duplicates('patient_id')
    med_pref = med_pref.rename(columns={'medication_label': 'recommended_medication_strategy'})
    patient_dx = patient_dx.merge(med_pref[['patient_id', 'recommended_medication_strategy']], on='patient_id', how='left')
    patient_dx['recommended_medication_strategy'] = patient_dx['recommended_medication_strategy'].fillna('recommend_maintain_regimen_with_monitoring')
else:
    def recommend_policy(row):
        if row['patient_category'] == 'high_complexity':
            return 'recommend_combo_therapy_plus_adherence_case_management'
        if row['patient_category'] == 'moderate_complexity':
            return 'recommend_adjust_dose_or_class_plus_behavioral_support'
        return 'recommend_maintain_regimen_with_monitoring'

    patient_dx['recommended_medication_strategy'] = patient_dx.apply(recommend_policy, axis=1)

dx_summary = dx_long.groupby('inferred_diagnosis').agg(
    n_rows=('patient_id', 'size'),
    n_patients=('patient_id', 'nunique')
).reset_index().sort_values('n_rows', ascending=False)

recommendation_summary = patient_dx.groupby(['patient_category', 'recommended_medication_strategy']).agg(
    n_patients=('patient_id', 'nunique'),
    mean_escalation_rate=('escalation_rate', 'mean'),
    mean_dx_burden=('dx_burden', 'mean')
).reset_index().sort_values(['patient_category', 'n_patients'], ascending=[True, False])

patient_dx.to_csv(TABLE_DIR / 'stage_l_patient_category_recommendations.csv', index=False)
dx_long.to_csv(TABLE_DIR / 'stage_l_multi_diagnosis_progression_long.csv', index=False)
dx_summary.to_csv(TABLE_DIR / 'stage_l_multi_diagnosis_summary.csv', index=False)
recommendation_summary.to_csv(TABLE_DIR / 'stage_l_medication_recommendation_summary.csv', index=False)

plt.figure(figsize=(9, 6))
plot_df = recommendation_summary.copy().sort_values('n_patients', ascending=False).head(9)
plt.barh(plot_df['patient_category'] + ' | ' + plot_df['recommended_medication_strategy'], plot_df['n_patients'])
plt.title('Stage L Patient Categories and Recommended Strategies')
plt.xlabel('Number of patients')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_l_patient_category_recommendations.png', dpi=140, bbox_inches='tight')
plt.close()

with open(REPORT_DIR / 'stage_l_recommendation_data_source.txt', 'w', encoding='utf-8') as f:
    f.write(f'registry_diagnosis_used: {diag_used}\n')
    f.write(f'registry_medication_used: {med_used}\n')
    f.write(f'diagnosis_rows: {len(diag_hist)}\n')
    f.write(f'medication_rows: {len(med_evt)}\n')

print('Multi-diagnosis progression and medication recommendation artifacts generated')
print('Registry diagnosis used:', diag_used, '| Registry medication used:', med_used)
recommendation_summary.head(10)

Multi-diagnosis progression and medication recommendation artifacts generated
Registry diagnosis used: True | Registry medication used: True


,patient_category,recommended_medication_strategy,n_patients,mean_escalation_rate,mean_dx_burden
3,high_complexity,escitalopram,62,0.083278,0.031600
4,high_complexity,fluoxetine,33,0.085343,0.027829
13,high_complexity,sertraline,33,0.082870,0.034014
0,high_complexity,aripiprazole_low,9,0.083900,0.034014
1,high_complexity,desvenlafaxine,5,0.081633,0.032653
2,high_complexity,duloxetine,4,0.086735,0.066327
6,high_complexity,lithium,4,0.086735,0.045918
5,high_complexity,lamotrigine,3,0.081633,0.040816
10,high_complexity,quetiapine,3,0.088435,0.068027
12,high_complexity,risperidone,3,0.081633,0.027211


In [14]:
# Stage L manifest + checklist proof
manifest_l = {
    'phase': 'L',
    'notebook': '09_stage_l_interpretability_ethics_clinical_trust.ipynb',
    'inputs': [
        'Data/metadata/phase_j_manifest.json',
        'Data/metadata/phase_k_manifest.json',
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet',
        'Data/diagnosis_history.parquet',
        'Data/medication_event.parquet'
    ],
    'outputs_tables': [
        'Results/tables/notebook09_stage_l/stage_l_model_metrics.csv',
        'Results/tables/notebook09_stage_l/stage_l_intrinsic_alignment_coefficients.csv',
        'Results/tables/notebook09_stage_l/stage_l_global_explanations.csv',
        'Results/tables/notebook09_stage_l/stage_l_local_explanations_long.csv',
        'Results/tables/notebook09_stage_l/stage_l_fairness_subgroup_audit.csv',
        'Results/tables/notebook09_stage_l/stage_l_failure_case_catalog.csv',
        'Results/tables/notebook09_stage_l/stage_l_counterfactual_examples.csv',
        'Results/tables/notebook09_stage_l/stage_l_patient_category_recommendations.csv',
        'Results/tables/notebook09_stage_l/stage_l_multi_diagnosis_progression_long.csv',
        'Results/tables/notebook09_stage_l/stage_l_multi_diagnosis_summary.csv',
        'Results/tables/notebook09_stage_l/stage_l_medication_recommendation_summary.csv'
    ],
    'outputs_figures': [
        'Results/figures/notebook09_stage_l/stage_l_global_explanation_barh.png',
        'Results/figures/notebook09_stage_l/stage_l_patient_category_recommendations.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook09_stage_l/stage_l_clinical_trust_summary.txt',
        'Results/reports/notebook09_stage_l/stage_l_explainability_runtime_log.txt',
        'Results/reports/notebook09_stage_l/stage_l_recommendation_data_source.txt',
        'Results/reports/notebook09_stage_l/stage_l_model_card_draft.md'
    ]
}
with open(META_DIR / 'phase_l_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_l, f, indent=4)

proof_l = {
    'model_metrics_generated': (TABLE_DIR / 'stage_l_model_metrics.csv').exists(),
    'global_explanations_generated': (TABLE_DIR / 'stage_l_global_explanations.csv').exists(),
    'local_explanations_generated': (TABLE_DIR / 'stage_l_local_explanations_long.csv').exists(),
    'fairness_audit_generated': (TABLE_DIR / 'stage_l_fairness_subgroup_audit.csv').exists(),
    'failure_catalog_generated': (TABLE_DIR / 'stage_l_failure_case_catalog.csv').exists(),
    'counterfactual_examples_generated': (TABLE_DIR / 'stage_l_counterfactual_examples.csv').exists(),
    'multi_diagnosis_generated': (TABLE_DIR / 'stage_l_multi_diagnosis_summary.csv').exists(),
    'medication_recommendations_generated': (TABLE_DIR / 'stage_l_medication_recommendation_summary.csv').exists(),
    'model_card_generated': (REPORT_DIR / 'stage_l_model_card_draft.md').exists(),
    'manifest_generated': (META_DIR / 'phase_l_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_l_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_l, 'explain_method': explain_method}, f, indent=4)

print('Stage L manifest/proof generated')
for k, v in proof_l.items():
    print('-', k, ':', v)
proof_l

Stage L manifest/proof generated
- model_metrics_generated : True
- global_explanations_generated : True
- local_explanations_generated : True
- fairness_audit_generated : True
- failure_catalog_generated : True
- counterfactual_examples_generated : True
- multi_diagnosis_generated : True
- medication_recommendations_generated : True
- model_card_generated : True
- manifest_generated : True


{'model_metrics_generated': True,
 'global_explanations_generated': True,
 'local_explanations_generated': True,
 'fairness_audit_generated': True,
 'failure_catalog_generated': True,
 'counterfactual_examples_generated': True,
 'multi_diagnosis_generated': True,
 'medication_recommendations_generated': True,
 'model_card_generated': True,
 'manifest_generated': True}

In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook09'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)